# Chapter 30 bridge: backpropagation against autograd

Chapter 30 derives every gradient of a one-hidden-layer network by hand and checks it numerically. Here the same weights and the same five examples go through PyTorch, and autograd's gradients are compared with the chapter's.

Nothing here is reimplemented. The chapter's own file is executed, its own weights and data are handed to PyTorch, and the chapter's hand-derived gradients are compared against autograd. Tolerances are stated per check and are **relative** to the size of the quantity being compared, because an absolute threshold means nothing without a scale.

Run order: top to bottom, from a fresh kernel. Requires `requirements-bridges.txt` on top of the book's own `requirements.txt`.

In [1]:
import os, sys, numpy as np, torch
torch.set_default_dtype(torch.float64)          # match NumPy's float64 exactly
CH = os.path.join("..", "code", "ch30")
os.chdir(CH) if os.path.basename(os.getcwd()) != "ch30" else None
def run(name):
    exec(open(name, encoding="utf-8").read(), globals())
run("_lib.py")
print("chapter:", os.path.basename(os.getcwd()), "| torch", torch.__version__, "| numpy", np.__version__)


chapter: ch30 | torch 2.14.0 | numpy 2.4.4


In [2]:
def report(name, ours, theirs, tol=1e-9):
    a = np.asarray(ours, dtype=float); b = np.asarray(theirs, dtype=float)
    denom = max(np.abs(b).max(), 1e-300)
    absd = np.abs(a - b).max(); rel = absd / denom
    ok = rel <= tol
    RESULTS.append(dict(check=name, max_abs=float(absd), max_rel=float(rel),
                        scale=float(denom), tol=tol, passed=bool(ok)))
    print(f"{'PASS' if ok else 'FAIL'}  {name:52s} max|diff| {absd:.3e}   "
          f"relative {rel:.2e}   (tolerance {tol:g})")
    return ok
RESULTS = []
MEASUREMENTS = []      # reported, never asserted: these have no single right answer


### The chapter's forward pass and hand-derived gradients
`c3.py` builds the network and `c4.py` derives the gradients.

In [3]:
run("c1.py"); run("c2.py"); run("c3.py"); run("c4.py")

pixel vector shape: (64,)
weighted sum z = -3.005
after ReLU,  a = 0.000

that is the entire computation one neuron performs.
a layer is many of these run in parallel; a network is
layers of them run in sequence.
W1 shape (2, 8), W2 shape (8, 1)
W1 @ W2 shape (2, 1)  <- collapses to one 2x1 matrix
two linear layers ARE one linear layer, just slower to compute.

linear model on a ring inside a ring: 0.495 accuracy
no straight line separates a ring from its centre, so a stack of
linear layers scores exactly what guessing scores. This is why
ReLU exists: it lets each layer bend the boundary, not just
rotate it.
input           (1, 64)
after layer 1   (1, 32)   (18 of 32 neurons fired)
after layer 2   (1, 10)
after softmax   (1, 10)   sums to 1.000000

predicted digit: 0   true label: 0
confidence in that digit: 0.156
     entry    analytic   numerical   match
(43, 7)   -0.023939   -0.023939    True
(27,21)    0.084851    0.084851    True
(43,16)   -0.009237   -0.009237    True

starting l

### The same computation in PyTorch
The chapter's `W1, b1, W2, b2` are copied into tensors; nothing is re-initialised.

In [4]:
tW1 = torch.tensor(W1, requires_grad=True); tb1 = torch.tensor(b1, requires_grad=True)
tW2 = torch.tensor(W2, requires_grad=True); tb2 = torch.tensor(b2, requires_grad=True)
tX = torch.tensor(Xb); tY = torch.tensor(Y)

tz1 = tX @ tW1 + tb1
ta1 = torch.relu(tz1)
tz2 = ta1 @ tW2 + tb2
# The chapter writes log(p + 1e-12) to keep the logarithm away from zero. torch's
# cross_entropy has no such guard, so the two differ by that epsilon. Compare like with
# like: build torch's loss with the chapter's own formula, guard included.
tp = torch.softmax(tz2, dim=1)
tloss = -(tY * torch.log(tp + 1e-12)).sum() / len(tX)
tloss.backward()
print(f"chapter loss {loss:.15f}   torch loss {tloss.item():.15f}")
report("forward: loss (same epsilon-guarded formula both sides)", loss, tloss.item(), 1e-12)

chapter loss 2.158043305785385   torch loss 2.158043305785385
PASS  forward: loss (same epsilon-guarded formula both sides) max|diff| 0.000e+00   relative 0.00e+00   (tolerance 1e-12)


np.True_

### Every hand-derived gradient against autograd

In [5]:
report("dW1  (input -> hidden weights)", dW1, tW1.grad.numpy())
report("db1  (hidden bias)",              db1, tb1.grad.numpy())
report("dW2  (hidden -> output weights)", dW2, tW2.grad.numpy())
report("db2  (output bias)",              db2, tb2.grad.numpy())

PASS  dW1  (input -> hidden weights)                       max|diff| 2.474e-12   relative 1.19e-11   (tolerance 1e-09)
PASS  db1  (hidden bias)                                   max|diff| 2.736e-12   relative 1.15e-11   (tolerance 1e-09)
PASS  dW2  (hidden -> output weights)                      max|diff| 4.136e-12   relative 1.75e-11   (tolerance 1e-09)
PASS  db2  (output bias)                                   max|diff| 2.447e-12   relative 2.18e-11   (tolerance 1e-09)


np.True_

### What the ReLU gradient does at exactly zero
The chapter uses `z1 > 0`, so a pre-activation of exactly zero gets gradient 0. PyTorch's `relu` makes the same choice. This is a real convention, not an accident, and it is worth seeing that the two agree.

In [6]:
z = torch.zeros(3, requires_grad=True)
torch.relu(z).sum().backward()
print("torch d(relu)/dz at z = 0:", z.grad.numpy(), " chapter's rule (z > 0):", (np.zeros(3) > 0).astype(float))
report("ReLU subgradient at exactly zero", (np.zeros(3) > 0).astype(float), z.grad.numpy(), 0.0)

torch d(relu)/dz at z = 0: [0. 0. 0.]  chapter's rule (z > 0): [0. 0. 0.]
PASS  ReLU subgradient at exactly zero                     max|diff| 0.000e+00   relative 0.00e+00   (tolerance 0)


np.True_

In [7]:
import json
n_pass = sum(1 for r in RESULTS if r["passed"])
print(f"\n{n_pass} of {len(RESULTS)} ASSERTED checks passed")
if MEASUREMENTS:
    print(f"{len(MEASUREMENTS)} reported measurement(s), not asserted:")
    for m in MEASUREMENTS: print("   ", m)
print(json.dumps(dict(checks=RESULTS, measurements=MEASUREMENTS), indent=1))
assert n_pass == len(RESULTS), "a gradient check failed"



6 of 6 ASSERTED checks passed
{
 "checks": [
  {
   "check": "forward: loss (same epsilon-guarded formula both sides)",
   "max_abs": 0.0,
   "max_rel": 0.0,
   "scale": 2.1580433057853847,
   "tol": 1e-12,
   "passed": true
  },
  {
   "check": "dW1  (input -> hidden weights)",
   "max_abs": 2.473771187894158e-12,
   "max_rel": 1.1916493816668938e-11,
   "scale": 0.2075922016955874,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "db1  (hidden bias)",
   "max_abs": 2.7361446441886983e-12,
   "max_rel": 1.1475347511329859e-11,
   "scale": 0.2384367568378425,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "dW2  (hidden -> output weights)",
   "max_abs": 4.1355530111530925e-12,
   "max_rel": 1.7523610648048613e-11,
   "scale": 0.23599890994003667,
   "tol": 1e-09,
   "passed": true
  },
  {
   "check": "db2  (output bias)",
   "max_abs": 2.4470980797275388e-12,
   "max_rel": 2.179848451522707e-11,
   "scale": 0.1122600095441565,
   "tol": 1e-09,
   "passed": true
  },
 